# HPC Scaling Analysis: Heat Diffusion Simulation

## Introduction

This notebook presents a **performance analysis** of a parallel Heat Diffusion simulation implemented using a hybrid **MPI + OpenMP** programming model.

The simulation solves the 2D heat equation on a grid using a stencil-based iterative approach. The goal is to evaluate how well the implementation scales across:

- **Threads** (OpenMP shared-memory parallelism)
- **Nodes** (MPI distributed-memory parallelism)

Three scaling experiments were conducted:

| Experiment | Parallelism | Grid Size | Variable |
|---|---|---|---|
| **Thread Scaling** | OpenMP | 15000×15000 (fixed) | 1 → 112 threads |
| **Weak Scaling** | MPI | 15000×15000 → 60000×60000 | 1 → 16 nodes |
| **Strong Scaling** | MPI | 15000×15000 (fixed) | 1 → 16 nodes |

All experiments were run on a **HPC cluster**, using up to **16 nodes** with **8 MPI tasks per node** and **14 OpenMP threads per task**.

The analysis focuses on:
- **Execution time** and **computation time**
- **Communication overhead**
- **Speedup** and **efficiency** relative to the baseline

# Thread Scaling Analysis

## Overview
The benchmark was run on a **single node** using OpenMP threads, with a **15000×15000 grid**, varying the number of threads from **1 to 112**.

In [1]:
import plotly.express as px
import pandas as pd


# Load the data from the CSV file
data: list[dict] = []
with open('results/results_threads.txt', 'r') as f:
    lines = f.readlines()
    
    for line in lines:
        if line.startswith('total'):
            pass
        elif line.startswith('execution_time'):
            pass
        elif line.strip() == '':
            pass
        else:
            values = line.split(',')
            data.append({
                'execution_time': float(values[0]),
                'computation_time': float(values[1]),
                'communication_time': float(values[2]),
                'threads': int(values[3]),
            })

df_thread = pd.DataFrame(data)


max_threads = df_thread['threads'].max()
x_range = [0, max_threads + 1]

# Asse x comune: tickt 1 2 4 8 16 32 56 84 112
# Asse x comuune: tick deve essere 1, 2, 4, 8, 16, 32, 56, 84, 112 (perché sono i thread che abbiamo usato)
#common_xaxis_thread = dict(tickmode='array', tick0=4, dtick=4, range=[4, max_threads + 1])#, tickvals=[4, 8, 16, 32, 56, 84, 112])
common_xaxis_thread = dict(autorange=True)
# Create a line plot for strong scaling
p_tot_threads = px.line(df_thread, x='threads', y='execution_time',
                         title='Total Execution Time', markers=True, hover_data={'computation_time', 'communication_time'})
p_tot_threads.update_layout(xaxis_title='Threads', yaxis_title='Execution Time (s)')
p_tot_threads.update_layout(title_x=0.5)
p_tot_threads.update_layout(plot_bgcolor='white')
p_tot_threads.update_layout(title=dict(subtitle=dict(text="Grid Size: 15000x15000, Threads: 14, Total Tasks: Nodes * 8")))
p_tot_threads.update_layout(
    xaxis=common_xaxis_thread,
    yaxis=dict(tickmode='linear', tick0=0, dtick=20, range=[0, df_thread['execution_time'].max() * 1.1])
)
desired_layout = p_tot_threads.layout


p_exec_threads = px.line(df_thread, x='threads', y='computation_time', markers=True)
p_exec_threads.update_layout(desired_layout)
p_exec_threads.update_layout(title=dict(text='Computation Time', x=0.5))
p_exec_threads.update_layout(xaxis_title='Threads', yaxis_title='Computation Time (s)')
p_exec_threads.update_layout(
    xaxis=common_xaxis_thread,
    yaxis=dict(tickmode='linear', tick0=0, dtick=20, range=[0, df_thread['computation_time'].max() * 1.1])
)

# moltiplica il valore dell'unità di tempo per 1e+9
df_thread['communication_time'] = df_thread['communication_time'] * 1000000
print(df_thread)


   execution_time  computation_time  communication_time  threads
0      338.035205        338.034626                38.0        1
1      180.348264        180.347686               134.0        2
2      117.386769        117.386307                32.0        4
3       90.946983         90.946514                35.0        8
4       71.790947         71.790295                54.0       16
5       36.119391         36.118795                41.0       32
6       20.705555         20.704967                40.0       56
7       14.793561         14.792761                37.0       84
8       11.410223         11.409407                48.0      112


In [2]:

p_comm_threads = px.line(df_thread, x='threads', y='communication_time', markers=True)
p_comm_threads.update_layout(desired_layout)
p_comm_threads.update_layout(title=dict(text='Communication Time', x=0.5))
p_comm_threads.update_layout(xaxis_title='Threads', yaxis_title='Communication Time (μs)')
p_comm_threads.update_layout(
    xaxis=common_xaxis_thread,
    yaxis=dict(autorange=True)
    #yaxis=dict(tickmode='linear', tick0=0, dtick=0.5, range=[0, df_thread['communication_time'].max() * 1.1])
)

p_comm_threads.show()



## Communication Time
- Communication overhead is **negligible** (measured in **microseconds**, ranging from ~38 μs to ~134 μs)
- No clear trend — variations are likely due to **thread synchronization noise**

In [3]:
p_exec_threads.show()
p_tot_threads.show()


## Execution & Computation Time
- At **1 thread**: execution time ≈ **338 s**
- At **112 threads**: execution time ≈ **11.4 s** (~**29.6× speedup**)
- Computation time closely tracks total execution time, confirming that the workload is **compute-bound**

In [4]:

p_speedup_threads = px.line(df_thread, x='threads', y=df_thread['execution_time'].iloc[0] / df_thread['execution_time'], markers=True)
p_speedup_threads.update_layout(desired_layout)
p_speedup_threads.update_layout(title=dict(text='Speedup', x=0.5))
p_speedup_threads.update_layout(xaxis_title='Threads', yaxis_title='Speedup (T1 / TN)')
p_speedup_threads.update_layout(
    xaxis=common_xaxis_thread,
    yaxis=dict(autorange=True) #dict(tickmode='linear', tick0=0, dtick=20, range=[0, df_thread['execution_time'].max() * 1.1])
    #yaxis=dict(tickmode='linear', tick0=10, dtick=10, range=[0, 100])
)
# Add ideal speedup line
ideal_speedup = df_thread['threads']
p_speedup_threads.add_scatter(x=df_thread['threads'], y=ideal_speedup, mode='lines', name='Ideal Speedup', line=dict(dash='dash', color='red'))
p_speedup_threads.show()


## Speedup
- The observed speedup at 112 threads is **~29.6×**, far below the **ideal linear speedup of 112×**
- This is expected due to:
    - **Thread synchronization overhead** at higher thread counts
- The speedup curve flattens notably after **32 threads**, suggesting diminishing returns beyond that point


## Efficiency
| Threads | Speedup | Efficiency |
|---------|---------|------------|
| 1       | 1.0×    | 100%       |
| 8       | 3.7×    | 46%        |
| 32      | 9.4×    | 29%        |
| 112     | 29.6×   | 26%        |

# Weak Scaling Analysis

## Overview
The benchmark was run on **1 to 16 nodes** using MPI, with a **grid that scales proportionally** with the number of nodes (starting from **15000×15000** up to **60000×60000**), keeping the **workload per process constant**.

In [5]:
# Load the data from the CSV file
data: list[dict] = []
with open('results/results_weak.txt', 'r') as f:
    lines = f.readlines()
    
    for line in lines:
        if line.startswith('total'):
            pass
        elif line.startswith('execution_time'):
            pass
        elif line.strip() == '':
            pass
        else:
            values = line.split(',')
            data.append({
                'execution_time': float(values[0]),
                'computation_time': float(values[1]),
                'communication_time': float(values[2]),
                'nodes': int(values[3]),
                'tasks': int(values[4]),
                'x_size': int(values[5]),
                'y_size': int(values[6]),
            })

df_weak = pd.DataFrame(data)
print(df_weak)


   execution_time  computation_time  communication_time  nodes  tasks  x_size  \
0       45.251847         42.919829            2.331688      1      8   15000   
1       45.509712         43.113820            2.395667      2     16   30000   
2       44.866397         42.782651            2.083520      4     32   30000   
3       45.403276         43.031002            2.372030      8     64   60000   
4       45.114921         42.308932            2.805807     16    128   60000   

   y_size  
0   15000  
1   15000  
2   30000  
3   30000  
4   60000  


In [6]:

max_threads = df_weak['nodes'].max()
x_range = [0, max_threads + 1]

# Asse x comune: tickt 1 2 4 8 16 32 56 84 112
# Asse x comuune: tick deve essere 1, 2, 4, 8, 16, 32, 56, 84, 112 (perché sono i thread che abbiamo usato)
#common_xaxis_thread = dict(tickmode='array', tick0=4, dtick=4, range=[4, max_threads + 1])#, tickvals=[4, 8, 16, 32, 56, 84, 112])
common_xaxis_thread = dict(autorange=True)
# Create a line plot for strong scaling
p_tot_weak = px.line(df_weak, x='nodes', y='execution_time',
                         title='Total Execution Time', markers=True, hover_data={'tasks', 'x_size', 'y_size'})
p_tot_weak.update_layout(xaxis_title='Nodes', yaxis_title='Execution Time (s)')
p_tot_weak.update_layout(title_x=0.5)
p_tot_weak.update_layout(plot_bgcolor='white')
p_tot_weak.update_layout(
    xaxis=common_xaxis_thread,
    yaxis=dict(tickmode='linear', tick0=0, dtick=20, range=[0, df_weak['execution_time'].max() * 1.1])
)
desired_layout = p_tot_weak.layout


p_exec_weak = px.line(df_weak, x='nodes', y='computation_time', markers=True, hover_data={'tasks', 'x_size', 'y_size'})
p_exec_weak.update_layout(desired_layout)
p_exec_weak.update_layout(title=dict(text='Computation Time', x=0.5))
p_exec_weak.update_layout(xaxis_title='Nodes', yaxis_title='Computation Time (s)')
p_exec_weak.update_layout(
    xaxis=common_xaxis_thread,
    yaxis=dict(tickmode='linear', tick0=0, dtick=20, range=[0, df_weak['computation_time'].max() * 1.1])
)


p_comm_weak = px.line(df_weak, x='nodes', y='communication_time', markers=True,
                      hover_data={'tasks', 'x_size', 'y_size'})
p_comm_weak.update_layout(desired_layout)
p_comm_weak.update_layout(title=dict(text='Communication Time', x=0.5))
p_comm_weak.update_layout(xaxis_title='Nodes', yaxis_title='Communication Time (s)')
p_comm_weak.update_layout(
    xaxis=common_xaxis_thread,
    #yaxis=dict(autorange=True)
    yaxis=dict(tickmode='linear', tick0=0, dtick=0.5, range=[0, df_weak['communication_time'].max() * 1.1])
)

p_comm_weak.show()

## Communication Time
- Communication overhead ranges from **~2.1 s to ~2.8 s**, representing a **small but non-negligible** fraction (~5%) of total execution time
- A slight increase at higher node counts is expected due to **increased inter-node halo exchange**
- The overhead remains **bounded**, indicating good scalability of the communication pattern

In [7]:

p_exec_weak.show()

## Computation Time
- Computation time remains **nearly constant** across all node counts (~**45 s**)
- This confirms **ideal weak scaling behavior**: doubling the nodes while doubling the problem size keeps the time stable

In [8]:

p_tot_weak.show()

## Total Execution Time
- The execution time is almost entirely made by **computation time** confirming the workload is **compute-bound**

In [9]:


p_speedup_weak = px.line(df_weak, x='nodes', y=df_weak['execution_time'].iloc[0] / df_weak['execution_time'], markers=True,
                         hover_data={'tasks', 'x_size', 'y_size'})
p_speedup_weak.update_layout(desired_layout)
p_speedup_weak.update_layout(title=dict(text='Speedup', x=0.5))
p_speedup_weak.update_layout(xaxis_title='Nodes', yaxis_title='Speedup (T1 / TN)')
p_speedup_weak.update_layout(
    xaxis=common_xaxis_thread,
    yaxis=dict(tickmode='linear', dtick=0.1, range=[0.8, 1.2])
    #yaxis=dict(autorange=True) #dict(tickmode='linear', tick0=0, dtick=20, range=[0, df_weak['execution_time'].max() * 1.1])
    #yaxis=dict(tickmode='linear', tick0=10, dtick=10, range=[0, 100])
)
# Add ideal speedup line
ideal_speedup = [1] * len(df_weak['nodes']) # Crea una lista di 1 con la stessa lunghezza di df_weak['threads']
p_speedup_weak.add_scatter(x=df_weak['nodes'], y=ideal_speedup, mode='lines', name='Ideal Speedup', line=dict(dash='dash', color='red'))
p_speedup_weak.show()

## Speedup and Efficiency
| Nodes | Execution Time (s) | Efficiency |
|-------|--------------------|------------|
| 1     | 45.25              | 100%       |
| 2     | 45.51              | 99.4%      |
| 4     | 44.87              | 100.8%     |
| 8     | 45.40              | 99.7%      |
| 16    | 45.11              | 100.3%     |

- Speedup and efficiency stays **very close to 100% of the baseline** across all configurations
- The slight variations (~±1%) are within **measurement noise**
- The implementation demonstrates **excellent weak scaling**

The communication time is non-zero even on a single node due to the intrinsic overhead of MPI primitives. The increase observed at two nodes is caused by the introduction of network latency, while the subsequent reduction is related to the decrease in the amount of data handled by each process in the strong scaling regime.

# Strong Scaling Analysis

## Overview
The benchmark was run on **1 to 16 nodes** using MPI, with a **fixed 15000×15000 grid** and always with **14 threads**, only varying the number of nodes from **1 to 16** (using **8 tasks per node**, for a total of up to **128 MPI tasks**).

In [10]:


# Load the data from the CSV file
data: list[dict] = []
with open('results/results_strong.txt', 'r') as f:
    lines = f.readlines()
    
    for line in lines:
        if line.startswith('total'):
            pass
        elif line.startswith('execution_time'):
            pass
        elif line.strip() == '':
            pass
        else:
            values = line.split(',')
            data.append({
                'execution_time': float(values[0]),
                'computation_time': float(values[1]),
                'communication_time': float(values[2]),
                'nodes': int(values[3]),
            })

df_strong = pd.DataFrame(data)
print(df_strong)


   execution_time  computation_time  communication_time  nodes
0       45.095692         43.178261            1.917230      1
1       23.518684         21.136468            2.381976      2
2       12.573552         10.738566            1.834615      4
3        7.006069          5.298784            1.707105      8
4        3.596636          2.566808            1.029596     16


In [11]:

max_nodes = df_strong['nodes'].max()
x_range = [0, max_nodes + 1]

# Asse x comune: tick da 2 in poi (niente "0" sull'asse x, così non si sovrappone allo "0" della y)
common_xaxis = dict(tickmode='linear', tick0=2, dtick=2)

# Create a line plot for strong scaling
p_tot_strong = px.line(df_strong, x='nodes', y='execution_time',
                         title='Total Execution Time', markers=True)
p_tot_strong.update_layout(xaxis_title='Nodes', yaxis_title='Execution Time (s)')
p_tot_strong.update_layout(title_x=0.5)
p_tot_strong.update_layout(plot_bgcolor='white')
p_tot_strong.update_layout(title=dict(subtitle=dict(text="Grid Size: 15000x15000, Threads: 14, Total Tasks: Nodes * 8")))
p_tot_strong.update_layout(
    xaxis=common_xaxis,
    yaxis=dict(tickmode='linear', tick0=0, dtick=5, range=[0, df_strong['execution_time'].max() * 1.1])
)
desired_layout = p_tot_strong.layout


p_exec_strong = px.line(df_strong, x='nodes', y='computation_time', markers=True)
p_exec_strong.update_layout(desired_layout)
p_exec_strong.update_layout(title=dict(text='Computation Time', x=0.5))
p_exec_strong.update_layout(xaxis_title='Nodes', yaxis_title='Computation Time (s)')
p_exec_strong.update_layout(
    xaxis=common_xaxis,
    yaxis=dict(tickmode='linear', tick0=0, dtick=5, range=[0, df_strong['computation_time'].max() * 1.1])
)


p_comm_strong = px.line(df_strong, x='nodes', y='communication_time', markers=True)
p_comm_strong.update_layout(desired_layout)
p_comm_strong.update_layout(title=dict(text='Communication Time', x=0.5))
p_comm_strong.update_layout(xaxis_title='Nodes', yaxis_title='Communication Time (s)')
p_comm_strong.update_layout(
    xaxis=common_xaxis,
    yaxis=dict(tickmode='linear', tick0=0, dtick=0.5, range=[0, df_strong['communication_time'].max() * 1.1])
)

p_comm_strong.show()



## Communication Time
- Communication time ranges from **~1.03 s** (16 nodes) to **~2.38 s** (2 nodes)
- Overhead is **non-negligible** but bounded, representing ~4–5% of total execution time
- A slight **decrease at higher node counts** is probably due to each process handling a **smaller portion of the grid**, reducing halo exchange volume.

In [12]:
p_exec_strong.show()

In [13]:
p_tot_strong.show()

## Computation & Total Execution Time
- At **1 node**: execution time ≈ **45.1 s**
- At **16 nodes**: execution time ≈ **3.6 s** (~**12.5× speedup**)
- Computation time closely tracks total execution time, **always** confirming the workload is **compute-bound**

In [14]:
p_speedup_strong = px.line(df_strong, x='nodes', y=df_strong['execution_time'].iloc[0] / df_strong['execution_time'], markers=True)
p_speedup_strong.update_layout(desired_layout)
p_speedup_strong.update_layout(title=dict(text='Speedup', x=0.5))
p_speedup_strong.update_layout(xaxis_title='Nodes', yaxis_title='Speedup (T1 / TN)')
p_speedup_strong.update_layout(
    xaxis=common_xaxis,
    yaxis=dict(tickmode='linear', tick0=0, dtick=1, range=[0, df_strong['execution_time'].iloc[0] / df_strong['execution_time'].min() * 1.1])
)
# Add ideal speedup line
ideal_speedup = df_strong['nodes']
p_speedup_strong.add_scatter(x=df_strong['nodes'], y=ideal_speedup, mode='lines', name='Ideal Speedup', line=dict(dash='dash', color='red'))
p_speedup_strong.show()



## Speedup
- The observed speedup at 16 nodes is **~12.5×**, below the **ideal linear speedup of 16×**

## Efficiency
| Nodes | Execution Time (s) | Speedup | Efficiency |
|-------|--------------------|---------|------------|
| 1     | 45.10              | 1.0×    | 100%       |
| 2     | 23.52              | 1.9×    | 95.9%      |
| 4     | 12.57              | 3.6×    | 89.7%      |
| 8     | 7.01               | 6.4×    | 80.5%      |
| 16    | 3.60               | 12.5×   | 78.3%      |

- The gradual efficiency decrease is expected as **communication overhead** becomes relatively more significant at higher node counts

## Conclusions

### Thread Scaling
- Increasing threads from **1 to 112** reduces execution time from **~338 s to ~11.4 s**, achieving a **~29.6× speedup**
- Efficiency drops significantly: from **100% at 1 thread** to **~26% at 112 threads**
- Speedup flattens after **32 threads**, indicating **diminishing returns** due to thread synchronization overhead
- Communication overhead is **negligible** (tens of microseconds), confirming the bottleneck is purely **computational**

### Strong Scaling 
- Increasing nodes from **1 to 16** reduces execution time from **~45.1 s to ~3.6 s**, achieving a **~12.5× speedup**
- Efficiency decreases gradually from **100% at 1 node** to **~78% at 16 nodes**, which is a **very good result**
- Communication overhead slightly **decreases** at higher node counts due to smaller per-process halo exchange volumes

### Overall
- The workload is consistently **compute-bound** across all scaling experiments
- **strong scaling** shows better efficiency than **thread scaling**, suggesting that distributed memory parallelism is better suited for this workload at large scales but 
at the current node count, the impact of the cluster topology on communication time is negligible. However, at larger scales, the physical distance between nodes and the network topology can significantly affect communication performance. In such cases, node cherry picking should be considered to minimize communication time.